In [36]:
from numpy.ma.core import indices

import Human_SleepSCoring.Tim.Sleep_Scripts.feature_visualizations as F
import mne
import Human_SleepSCoring.Tim.DeepNeuralNetworkSleep.hdf5_files.Artefacts_Detection as A
import pickle
from joblib import Parallel, delayed
from scipy.signal import hilbert
import os
import numpy as np
import pandas as pd


In [37]:
# Human frequency bands (AASM)
delta_band = [0.5,4]
theta_band = [4,8]
alpha_band = [8,13]
sigma_band = [12,14] # as per Gennaro & Ferrara Sleep spindles: an overview 2003
beta_band = [13,30]
gamma_band = [30,90] # as per Buzsaki & Wang Mechanisms of Gamma Oscillations 2012
noise_band = [0,0.5]
# total_band = [0,90]

In [38]:
def psd_multitaper(lfp_data, fs, frequency_band, window_length):
    all_power_sum = []

    # loop through each segment
    for start in range(0, len(lfp_data) - window_length + 1, window_length):
        window = lfp_data[start:min(start + window_length, len(lfp_data))]

        # compute power spectral density using multitaper method
        psd, freqs = mne.time_frequency.psd_array_multitaper(window, fs, fmin=frequency_band[0], fmax=frequency_band[1],
                                                             n_jobs=1, verbose='warning')

        # compute total power within frequency band
        freq_indices = (freqs >= frequency_band[0]) & (freqs <= frequency_band[1])
        curr_sum = np.sum(psd)
        all_power_sum.append(curr_sum)

    return all_power_sum


def wei_normalizing(data):
    data = np.array(data)

    bottom = data[data <= np.nanpercentile(data, 10, axis=0)]
    top = data[data >= np.nanpercentile(data, 90, axis=0)]

    bottom_avg = np.average(bottom) if len(bottom) > 0 else 0
    top_avg = np.average(top) if len(top) > 0 else 1

    denom = top_avg - bottom_avg if top_avg != bottom_avg else 1
    normalized_data = (data - bottom_avg) / denom
    normalized_data = np.clip(normalized_data, 0.05, 1)

    return normalized_data


def fragment_join(subject, night, type):
    base = f"D:/EEG_Data_stage/{subject}/iEEG/{type}"
    score_base = f"D:/EEG_Data_stage/{subject}/iEEG/U_sleep_API_10s"
    files = []
    scores = []
    for file in os.listdir(base):
        if f"night{night}" in file and ".vhdr" in file:
            files.append(os.path.join(base, file))
            scores.append(os.path.join(score_base, file.replace(".vhdr", "_hypnogram.npy")))
    return files, scores


def average_means_dicts(*means_dicts):
    """Average multiple nested mean dictionaries together (handles NaNs)."""
    all_states = means_dicts[0]['index_means'].keys()
    averaged = {}

    for state in all_states:
        averaged[state] = {}
        for key in means_dicts[0]['index_means'][state].keys():
            values = [
                m['index_means'][state][key]
                for m in means_dicts
                if not np.isnan(m['index_means'][state][key])
            ]
            averaged[state][key] = np.mean(values) if values else np.nan
    return averaged

def average_per_index_per_state(index_vals, hypnogram):
    """
    Compute mean index values per sleep state, organized by index.

    Parameters
    ----------
    index_vals : dict
        Dict of index_name -> numpy array of per-epoch values.
        Example: {"W": arr1, "N": arr2, "R": arr3, "1": arr4, ...}
    hypnogram : np.ndarray
        Array of integers (0–4) indicating sleep states per epoch.

    Returns
    -------
    dict
        {index_name: {"Wake": val, "N1": val, "N2": val, "N3": val, "REM": val}}
    """
    # Sleep stage label map
    state_labels = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM"}

    # Ensure consistent lengths
    min_len = min(len(hypnogram), *[len(v) for v in index_vals.values()])
    hypnogram = hypnogram[:min_len]

    # Trim and prepare output
    averaged = {}

    for idx_name, values in index_vals.items():
        values = np.asarray(values[:min_len])
        state_means = {}
        for s_int, s_name in state_labels.items():
            mask = (hypnogram == s_int)
            vals = values[mask]
            state_means[s_name] = np.nanmean(vals) if len(vals) > 0 else np.nan
        averaged[idx_name] = state_means

    return averaged

In [39]:
def prepare_aperiodic_avg_data(dicts):
    """
    Prepare per-subject aperiodic averages for plotting.
    Returns a DataFrame with columns: ['Subject', 'State', 'Aperiodic']
    and a structure ready for F.plot_averaged_aperiodic_violin().
    """
    data_rows = []
    for i, d in enumerate(dicts):
        if 'aperiodic_avg' not in d:
            continue  # skip subjects that lack the new data
        subj_name = f"Subj_{i+1}"
        for state, value in d['aperiodic_avg'].items():
            data_rows.append({'Subject': subj_name, 'State': state, 'Aperiodic': value})

    df = pd.DataFrame(data_rows)

    # Compute group means per state
    grouped = df.groupby('State')['Aperiodic'].mean().reset_index()

    # Prepare structure for plotting function
    averaged_aperiodic = {
        'df_plot': df,  # per-subject data
        'group_means': grouped,  # group-level averages
        'labels': grouped['State'].tolist(),
        'data_for_violin': [
            df.loc[df['State'] == s, 'Aperiodic'].dropna().values
            for s in grouped['State']
        ],
        'all_states': list(range(len(grouped['State'])))
    }
    return averaged_aperiodic

def prepare_dfa_avg_data(dicts):
    """
    Prepare per-subject aperiodic averages for plotting.
    Returns a DataFrame with columns: ['Subject', 'State', 'Aperiodic']
    and a structure ready for F.plot_averaged_aperiodic_violin().
    """
    data_rows = []
    for i, d in enumerate(dicts):
        if 'dfa_avg' not in d:
            continue  # skip subjects that lack the new data
        subj_name = f"Subj_{i+1}"
        for state, value in d['dfa_avg'].items():
            data_rows.append({'Subject': subj_name, 'State': state, 'Dfa': value})

    df = pd.DataFrame(data_rows)

    # Compute group means per state
    grouped = df.groupby('State')['Dfa'].mean().reset_index()

    # Prepare structure for plotting function
    averaged_dfa = {
        'df_plot': df,  # per-subject data
        'group_means': grouped,  # group-level averages
        'labels': grouped['State'].tolist(),
        'data_for_violin': [
            df.loc[df['State'] == s, 'Dfa'].dropna().values
            for s in grouped['State']
        ],
        'all_states': list(range(len(grouped['State'])))
    }
    return averaged_dfa

# --- Safe log-transform and normalization helper ---
def safe_log_normalize(arr):
    """Apply log transform and normalization safely."""
    arr = np.asarray(arr, dtype=float)
    # Avoid log(0) or negatives
    arr = np.where(arr <= 0, np.nanmin(arr[arr > 0]) * 0.1 if np.any(arr > 0) else 1e-6, arr)
    arr = np.log(arr)
    return wei_normalizing(arr)

def prepare_mse_avg_data(dicts):
    """
    Prepare per-subject MSE averages for plotting.
    Returns a dictionary containing:
        - df_plot: per-subject DataFrame
        - group_means: group-level averages
        - labels: list of state labels
        - data_for_violin: per-state data arrays for plotting
        - all_states: list of indices for all states
    """
    data_rows = []
    
    for i, d in enumerate(dicts):
        if 'mse_avg' not in d or not isinstance(d['mse_avg'], dict):
            continue  # skip subjects without mse_avg or invalid format
        
        subj_name = f"Subj_{i+1}"
        for state, value in d['mse_avg'].items():
            if state is None:
                continue  # skip if state is missing
            data_rows.append({
                'subject': subj_name,
                'state': state,
                'mse': value
            })
    
    if not data_rows:
        # Return empty structure if no valid data
        return {
            'df_plot': pd.DataFrame(columns=['subject', 'state', 'mse']),
            'group_means': pd.DataFrame(columns=['state', 'mse']),
            'labels': [],
            'data_for_violin': [],
            'all_states': []
        }
    
    df = pd.DataFrame(data_rows)
    
    # Compute group means per state
    grouped = df.groupby('state')['mse'].mean().reset_index()
    
    # Prepare structure for plotting
    averaged_mse = {
        'df_plot': df,
        'group_means': grouped,
        'labels': grouped['state'].tolist(),
        'data_for_violin': [
            df.loc[df['state'] == s, 'mse'].dropna().values
            for s in grouped['state']
        ],
        'all_states': list(range(len(grouped['state'])))
    }
    
    return averaged_mse

In [40]:
def avg_plots(raw, hpc_channel, pfc_channel, states, output_dir, epoch_length, fs, window_length, subject, night):
    """
    Compute or load PSDs and index values per state for a subject/night.
    Saves PSDs to a cache file to avoid recomputation.
    """

    
    # Bandpass EMG
    raw.filter(l_freq=10, h_freq=70, picks='EMG1-EMG2')
    hpc_data = np.ravel(raw.get_data(picks=hpc_channel)[0])
    pfc_data = np.ravel(raw.get_data(picks=pfc_channel)[0])
    EMG = raw.get_data(picks='EMG1-EMG2')[0]
    EMG = EMG[:len(EMG)//(epoch_length * fs) * (epoch_length * fs)] 
    EMG = EMG.reshape(-1, (epoch_length * fs))
    EMG = EMG.sum(axis=1)
    EMG = abs(hilbert(EMG))

    amp_thresh = [6, 4]
    time_win_thresh = [0.2, 0.1]

    raw_hpc = A.removeArtefacts(hpc_data, 250, amp_thresh, time_win_thresh)[0]
    raw_pfc = A.removeArtefacts(pfc_data, 250, amp_thresh, time_win_thresh)[0]
    # --- Set up cache directory ---
    cache_dir = os.path.join(output_dir, "cached_psd")
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"subject_{subject}_night_{night}_psd.pkl")

    # --- Try loading cached PSD data ---
    if os.path.exists(cache_file):
        print(f"✅ Loading cached PSDs for subject {subject}, night {night}...")
        with open(cache_file, "rb") as f:
            noise, delta, theta, sigma, beta, gamma, alpha = pickle.load(f)
    else:
        print(f"⚙️ Computing PSDs for subject {subject}, night {night}...")

        # PSD computation
        raws = [
            raw_pfc, raw_pfc, raw_hpc, raw_pfc, raw_pfc, raw_pfc, raw_hpc
        ]
        fr_bands = [noise_band, delta_band, theta_band, sigma_band, beta_band, gamma_band, alpha_band]

        noise, delta, theta, sigma, beta, gamma, alpha = Parallel(n_jobs=7)(
            delayed(psd_multitaper)(raw_sig, fs, band, window_length)
            for raw_sig, band in zip(raws, fr_bands)
        )

        # Save PSDs to cache
        with open(cache_file, "wb") as f:
            pickle.dump((noise, delta, theta, sigma, beta, gamma, alpha), f)
        print(f"💾 PSDs saved to cache: {cache_file}")

    # --- Normalize and smooth PSDs ---
    noise_norm = wei_normalizing(noise)
    delta_norm = wei_normalizing(delta)
    theta_norm = wei_normalizing(theta)
    sigma_norm = wei_normalizing(sigma)
    beta_norm = wei_normalizing(beta)
    gamma_norm = wei_normalizing(gamma)
    alpha_norm = wei_normalizing(alpha)
    EMG_norm = wei_normalizing(EMG)

    def smooth(x, n=5):
        return np.convolve(np.convolve(np.convolve(x, np.ones(n)/n, mode='same'),
                                       np.ones(n)/n, mode='same'),
                           np.ones(n)/n, mode='same')

    noise_smoothed = smooth(noise_norm)
    delta_smoothed = smooth(delta_norm)
    theta_smoothed = smooth(theta_norm)
    sigma_smoothed = smooth(sigma_norm)
    beta_smoothed = smooth(beta_norm)
    gamma_smoothed = smooth(gamma_norm)
    alpha_smoothed = smooth(alpha_norm)
    EMG = np.convolve(EMG, np.ones(20*fs)/(20*fs), mode='same')

    # --- Index calculations ---
    index_n = F.index_N(delta_norm, alpha_norm, EMG_norm, 
                        np.ravel(raw.get_data(picks='EOG1')[0]), 
                        np.ravel(raw.get_data(picks='EOG2')[0]), 
                        epoch_length, fs)
    index_r = F.index_R(delta_norm, sigma_norm, EMG_norm, 
                        np.ravel(raw.get_data(picks='EOG1')[0]), 
                        np.ravel(raw.get_data(picks='EOG2')[0]), 
                        epoch_length, fs)
    index_w = F.index_W(theta_norm, gamma_norm, EMG_norm)

    # Align lengths
    min_len = min(len(states), len(index_w), len(index_n), len(index_r))
    print(f"{min_len}- min len")
    mapped_scores = np.ravel(states)[:min_len]
    # --- Trim all indices to same minimum length ---
    min_len = min(
        len(index_w),
        len(index_n),
        len(index_r),
        len(F.Index_1(delta_norm, gamma_norm, EMG_norm)),
        len(F.Index_2(delta_norm, theta_norm, sigma_norm)),
        len(F.Index_3(delta_norm, theta_norm, gamma_norm)),
        len(F.Index_4(delta_norm, theta_norm))
    )
    
    index_w = index_w[:min_len]
    index_n = index_n[:min_len]
    index_r = index_r[:min_len]
    index_1 = F.Index_1(delta_norm, gamma_norm, EMG_norm)[:min_len]
    index_2 = F.Index_2(delta_norm, theta_norm, sigma_norm)[:min_len]
    index_3 = F.Index_3(delta_norm, theta_norm, gamma_norm)[:min_len]
    index_4 = F.Index_4(delta_norm, theta_norm)[:min_len]
    
    # --- Apply to all indices ---
    index_w = safe_log_normalize(index_w)
    index_n = safe_log_normalize(index_n)
    index_r = safe_log_normalize(index_r)
    index_1 = safe_log_normalize(index_1)
    index_2 = safe_log_normalize(index_2)
    index_3 = safe_log_normalize(index_3)
    index_4 = safe_log_normalize(index_4)


   
    # Compute index means per state
    index_means = F.extract_index_values_per_state(
        index_n, index_r, index_w, mapped_scores
    )

    # --- Compute aperiodic exponents ---
    normalized_exponents, _, valid_states, _ = F.aperiodic_fit(pfc_data, states, fs, raw_pfc, output_dir)

    # Prepare per-window data for plotting
    df_plot, data_for_violin, all_states, labels = F.prepare_aperiodic_violin_data(valid_states, normalized_exponents)

    # --- Compute one average value per state for this subject ---
    aperiodic_avg = {}
    for i, state in enumerate(all_states):
        vals = data_for_violin[state]
        aperiodic_avg[labels[i]] = np.nanmean(vals) if len(vals) > 0 else np.nan
        
    dfa_plot, dfa_data_for_violin, dfa_all_states, dfa_labels, normalized_dfa = F.prepare_dfa_violin_data(states, fs, epoch_length, np.ravel(pfc_data))
    # --- Compute one average value per state for this subject ---
    dfa_avg = {}
    for i, state in enumerate(dfa_all_states):
        vals = dfa_data_for_violin[state]
        dfa_avg[dfa_labels[i]] = np.nanmean(vals) if len(vals) > 0 else np.nan
        
    mse_plot, mse_data_for_violin, mse_all_states, mse_labels, normalized_mse = F.prepare_mse_violin_data(states, fs, epoch_length, np.ravel(pfc_data))
    # --- Compute one average value per state for this subject ---
    mse_avg = {}
    for i, state in enumerate(mse_all_states):
        vals = mse_data_for_violin[state]
        mse_avg[mse_labels[i]] = np.nanmean(vals) if len(vals) > 0 else np.nan

    # --- Combine everything into a single features dictionary ---
    features = {
        'index_means': index_means,
        'aperiodic_violin': {
            'df_plot': df_plot,
            'data_for_violin': data_for_violin,
            'all_states': all_states,
            'labels': labels
        },
        'dfa_violin': {
            'df_plot': dfa_plot,
            'data_for_violin': dfa_data_for_violin,
            'all_states': dfa_all_states,
            'labels': dfa_labels
        },
        'mse_violin': {
            'df_plot': mse_plot,
            'data_for_violin': mse_data_for_violin,
            'all_states': mse_all_states,
            'labels': mse_labels
        },
        'mse_avg':mse_avg,
        'aperiodic_avg': aperiodic_avg,
        'dfa_avg':dfa_avg,
        "index_vals":{
            "W":smooth(index_w), "N":smooth(index_n), "R":smooth(index_r), "1": smooth(index_1),
            "2": smooth(index_2), "3": smooth(index_3), "4": smooth(index_4)
        },
        "aperiodic_fit": smooth(normalized_exponents),
        "dfa": smooth(normalized_dfa),
        "mse": smooth(normalized_mse),
        "noise": noise_smoothed,
        "theta": theta_smoothed,
        "delta": delta_smoothed
    }

    # Save features to file
    feature_file = os.path.join(output_dir, f"features_{subject}_night{night}.npy")
    np.save(feature_file, features)
    print(f"Saved features for {subject}_night{night} to {feature_file}")

    return features
    

In [41]:
# Extra-cranial
electrodes = {
    "2": ["Oz-Cz", "C3-Cz"],
    "7": ["Oz-Cz", "C3-Cz"],
    "15": ["Oz-Cz", "C3-Cz"], 
    "28": ["Oz-Cz", "C3-Cz"], 
    "31": ["Oz-Cz", "C3-Cz"],
    "63": ["Oz-Cz", "C3-Cz"],
    "67": ["Oz-Cz", "C3-Cz"], 
    "69": ["Oz-Cz", "C3-Cz"], 
    "84": ["Oz-Cz", "C3-Cz"], 
    "85": ["Oz-Cz", "C3-Cz"],
    "86": ["Oz-Cz", "C3-Cz"],
    "87": ["C4-Cz", "C3-Cz"],
    "132": ["C4-Cz", "C3-Cz"], 
    "134": ["F3", "C4"], 
    "135": ["F3", "C4"]
}

In [42]:
fs = 250
epoch_length = 10 
window_length = epoch_length * fs
base_dir = "D:/EEG_Data_stage"
output_dir = os.path.join(base_dir, "averaged_plots_ec")
cache_dir = os.path.join(base_dir, "cache")
os.makedirs(output_dir, exist_ok=True)


In [34]:
# Ensure the features directory exists
features_dir = os.path.join(base_dir, "features")
os.makedirs(features_dir, exist_ok=True)
indices_total_bar = {}

for file in os.listdir(base_dir):
    if file in electrodes.keys():
        print(f"-----------------------------------Adding subject: {file}-----------------------------------")
        channels = electrodes[file]

        # Handle "28" special case
        nights = [2] if "28" in file else [1, 2]

        for night in nights:
            subject_night = f"{file}_night{night}"
            feature_file = os.path.join(features_dir, f"features_{subject_night}.npy")

            # Skip if the file already exists
            if os.path.exists(feature_file):
                print(f"Skipping {subject_night}, feature file already exists.")
                continue

            print(f"Converting night: {night}")
            subject = file
            night_str = f"{night}"
            file_type = "converted_ec"

            hpc_channel = channels[0]
            pfc_channel = channels[1]

            files, scores_files = fragment_join(subject, night_str, file_type)
            raw_list = [mne.io.read_raw_brainvision(f, preload=True) for f in files]
            raw = mne.concatenate_raws(raw_list)
            score_list = [np.load(f) for f in scores_files]
            states = np.concatenate(score_list)

            # Compute your feature
            features = avg_plots(raw, hpc_channel, pfc_channel, states, cache_dir, epoch_length, fs, window_length, file, night)

            # Save it
            np.save(feature_file, features)
            print(f"Saved features for {subject_night} to {feature_file}")

-----------------------------------Adding subject: 67-----------------------------------
Skipping 67_night1, feature file already exists.
Skipping 67_night2, feature file already exists.
-----------------------------------Adding subject: 69-----------------------------------
Skipping 69_night1, feature file already exists.
Skipping 69_night2, feature file already exists.
-----------------------------------Adding subject: 84-----------------------------------
Skipping 84_night1, feature file already exists.
Skipping 84_night2, feature file already exists.
-----------------------------------Adding subject: 85-----------------------------------
Skipping 85_night1, feature file already exists.
Skipping 85_night2, feature file already exists.
-----------------------------------Adding subject: 132-----------------------------------
Skipping 132_night1, feature file already exists.
Skipping 132_night2, feature file already exists.
-----------------------------------Adding subject: 134--------

In [43]:
# --- Load feature files ---
features_dir = os.path.join(base_dir, "features")
all_feature_files = [
    f for f in os.listdir(features_dir)
    if f.startswith("features_") and f.endswith(".npy")
]

loaded_dicts = []
for file_name in all_feature_files:
    file_path = os.path.join(features_dir, file_name)
    data = np.load(file_path, allow_pickle=True).item()
    loaded_dicts.append(data)

print(f"✅ Loaded {len(loaded_dicts)} feature files from {features_dir}")

for diction in loaded_dicts:
    states = diction["aperiodic_violin"]["df_plot"].iloc[:,0].to_numpy()
    averaged_indices = average_per_index_per_state(diction["index_vals"], states)

# --- Combine and average everything ---
if loaded_dicts:
    averaged_index_means = average_means_dicts(*loaded_dicts)
    averaged_aperiodic_violin = prepare_aperiodic_avg_data(loaded_dicts)
    averaged_dfa = prepare_dfa_avg_data(loaded_dicts)
    averaged_mse = prepare_mse_avg_data(loaded_dicts)

    averaged_features = {
        'index_means': averaged_index_means,
        'aperiodic_violin': averaged_aperiodic_violin,
        'dfa_violin': averaged_dfa,
        'mse_avg': averaged_mse,
        'index_avg': averaged_indices
    }

    print("✅ Averaged features prepared for plotting.")
else:
    averaged_features = {}
    print("⚠️ No feature files found to average.")


✅ Loaded 29 feature files from D:/EEG_Data_stage\features
✅ Averaged features prepared for plotting.


In [50]:
print(averaged_features["dfa_violin"].keys())

{'df_plot':      Subject State       Dfa
0     Subj_1     W -0.288480
1     Subj_1    N1 -0.340699
2     Subj_1    N2 -0.454899
3     Subj_1    N3  0.008743
4     Subj_1   REM -0.441807
..       ...   ...       ...
140  Subj_29     W -0.152861
141  Subj_29    N1  0.137705
142  Subj_29    N2  0.302207
143  Subj_29    N3  0.651754
144  Subj_29   REM  0.055564

[145 rows x 3 columns], 'group_means':   State       Dfa
0    N1 -0.238964
1    N2 -0.070147
2    N3  0.397539
3   REM -0.134303
4     W -0.229012, 'labels': ['N1', 'N2', 'N3', 'REM', 'W'], 'data_for_violin': [array([-0.34069888, -0.52200058, -0.47001056,  0.60759358, -0.15764138,
       -0.18353172, -0.38977364, -0.29920086, -0.24522381, -0.41299985,
       -0.37047742, -0.36299116, -0.51564698, -0.4789865 , -0.49153913,
       -0.3004607 , -0.31207155,  0.18483431, -0.33110742, -0.20719381,
        0.61845714, -0.28098208, -0.05434358, -0.53777002, -0.4406791 ,
       -0.20602314, -0.32823657,  0.13770455]), array([-0.45489926, -

In [ ]:
# Plots averaged by subject
F.plot_index_barplot(averaged_features['index_means'], output_dir)

F.plot_averaged_aperiodic_violin(averaged_features['aperiodic_violin'], output_dir)

F.plot_averaged_dfa_violin(averaged_features['dfa_violin'], output_dir)

F.plot_averaged_mse_violin(averaged_features['mse_avg'], output_dir)